<a href="https://colab.research.google.com/github/abdulhadi2005ag-cmd/flyrank-ml-internship-hadii/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdulhadi2005ag-cmd/flyrank-ml-internship-hadii/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

My label (same proxy as w03/w04): is_declining = 1 if trend_direction == "down",
else 0. This is a yes/no target with an observed value on every row, so per the
skill's method table I start with **Logistic Regression, then Random Forest** —
readable first, stronger second — rather than jumping straight to an ensemble.

My lane (Lane 2, refresh scoring) is really a ranking task: an editor works down
a priority list, not a per-page yes/no. So both models are trained as
classifiers (predict P(declining)) but graded as rankers — I sort the test set
by predicted probability and score **precision@50**, the same success metric I
picked in w02 and the same shape of question my w04 baseline rule answers.

Features: I reuse the honest, leakage-checked feature set from
w03_feature_leakage_check — all candidate numeric + categorical columns MINUS
impressions_last_30d/prev_30d and their clicks/sessions siblings (confirmed in
w03 to reconstruct trend_pct almost exactly) and minus the label family
(trend_direction, trend_pct) and the pseudonymous IDs. 38 columns after
one-hot encoding content_type / main_intent / competition_level and adding the
three has_*_data flags.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/abdulhadi2005ag-cmd/flyrank-ml-internship-hadii"
REPO_DIR = "flyrank-ml-internship-hadii"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

RANDOM_SEED = 1  # fixed everywhere below for reproducibility

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Loaded:", df.shape)

label = (df["trend_direction"] == "down").astype(int)
print("Label base rate (declining share, full dataset):", round(label.mean(), 3))

work = df.copy()
work["has_search_volume_data"] = work["search_volume"].notna().astype(int)
work["has_word_count_data"] = work["word_count"].notna().astype(int)
work["has_position_data"] = (work["avg_position"] > 0).astype(int)

# Same honest candidate list as w03_feature_leakage_check, with the two
# sibling-leak windows (impressions_last_30d/prev_30d + their clicks/sessions
# pairs) already removed — confirmed leaky there, not re-tested here.
numeric_candidates = [
    "content_age_days", "days_since_last_update", "word_count", "char_count",
    "search_volume", "competition", "cpc",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]

X_numeric = work[numeric_candidates].fillna(0)
X_numeric = pd.concat(
    [X_numeric, work[["has_search_volume_data", "has_word_count_data", "has_position_data"]]],
    axis=1,
)

categorical_candidates = ["content_type", "main_intent", "competition_level"]
X_categorical = pd.get_dummies(work[categorical_candidates], dummy_na=True, prefix=categorical_candidates)

X = pd.concat([X_numeric, X_categorical], axis=1)
groups = work["client_id"]

print("Feature matrix shape:", X.shape)
print("Columns:", list(X.columns))

Loaded: (30000, 44)
Label base rate (declining share, full dataset): 0.542
Feature matrix shape: (30000, 38)
Columns: ['content_age_days', 'days_since_last_update', 'word_count', 'char_count', 'search_volume', 'competition', 'cpc', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'has_search_volume_data', 'has_word_count_data', 'has_position_data', 'content_type_comparison article', 'content_type_feedly article', 'content_type_keyword article', 'content_type_nan', 'main_intent_commercial', 'main_intent_informational', 'main_intent_navigational', 'main_intent_transactional', 'main_intent_nan', 'competition_level_HIGH', 'competition_level_LOW', 'competition_level_MEDIUM', 'competition_level_nan']


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Grouped by client_id, not a plain random split. Two reasons:

1. The data skill flags client_id as a grouping key, never a feature — for the
   same reason, it has to be a grouping key for the SPLIT too. Pages from the
   same client share a CMS, an editorial voice, and a traffic baseline. A
   random row-level split would put pages from the same client on both sides,
   letting the model partly memorize "how client X's pages behave" instead of
   learning a pattern that generalizes to a client it has never seen.
2. My baseline (w04) and my real deployment target are the same: score pages
   for clients, some of whom the model has limited or no history for. A
   client-grouped split is the honest stand-in for that.

I use GroupShuffleSplit, 70/30, seed=1, split on client_id. Verified below:
zero client overlap between train and test.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(X, label, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = label.iloc[train_idx], label.iloc[test_idx]

print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Train clients:", groups.iloc[train_idx].nunique(), "| Test clients:", groups.iloc[test_idx].nunique())
overlap = set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])
print("Client overlap between train/test (must be 0):", len(overlap))
print("Test-set base rate:", round(y_test.mean(), 3), "(vs full-data 0.542 — some client-mix drift is expected)")

Train: (25969, 38) | Test: (4031, 38)
Train clients: 22 | Test clients: 10
Client overlap between train/test (must be 0): 0
Test-set base rate: 0.599 (vs full-data 0.542 — some client-mix drift is expected)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Same test split, same label, one new metric layered on top: precision@50,
computed identically for the baseline rule and both models — rank the test
set by score, take the top 50, check what share are actually declining.

The baseline's score (from w04) only fires when all three gates pass (stale,
has_demand, weak_ctr), so on this test split it only flags 19 of 4,031 rows —
the other 31 "top 50" slots are score=0 ties with no real ordering, which
makes precision@50 slightly generous to the baseline (it's really being
scored on 19 confident picks padded with 31 coin flips). So I report both
numbers: precision@50 (the metric everyone's compared on) AND precision at
the baseline's own flagged count (19), which is the fairer like-for-like
comparison.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
def precision_at_k(scores, y_true, k):
    order = np.argsort(-scores)[:k]
    return y_true.iloc[order].mean()

y_test_r = y_test.reset_index(drop=True)

# --- Baseline rule, scored on the test split only (same logic as w04) ---
test_df = df.iloc[test_idx].reset_index(drop=True).copy()
clean = df[df["avg_position"] > 0]
tier_median_ctr = clean.groupby("position_tier", observed=True)["ctr"].median()

test_df["has_position_data"] = test_df["avg_position"] > 0
test_df["tier_median_ctr"] = test_df["position_tier"].map(tier_median_ctr)
test_df["stale"] = test_df["days_since_last_update"] >= 90
test_df["has_demand"] = test_df["impressions_90d"] >= 500
test_df["weak_ctr"] = test_df["has_position_data"] & (test_df["ctr"] < test_df["tier_median_ctr"])
flagged = test_df["stale"] & test_df["has_demand"] & test_df["weak_ctr"]
test_df["baseline_score"] = np.where(flagged, test_df["impressions_90d"], 0)
n_flagged = int(flagged.sum())

p50_baseline = precision_at_k(test_df["baseline_score"].values, y_test_r, 50)
p_at_flagged_baseline = precision_at_k(test_df["baseline_score"].values, y_test_r, n_flagged)

# --- Logistic Regression ---
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

lr = LogisticRegression(max_iter=2000, random_state=RANDOM_SEED)
lr.fit(X_train_s, y_train)
lr_proba = lr.predict_proba(X_test_s)[:, 1]
lr_auc = roc_auc_score(y_test, lr_proba)
p50_lr = precision_at_k(lr_proba, y_test_r, 50)
p_at_flagged_lr = precision_at_k(lr_proba, y_test_r, n_flagged)

# --- Random Forest ---
rf = RandomForestClassifier(
    n_estimators=300, max_depth=8, min_samples_leaf=20,
    random_state=RANDOM_SEED, n_jobs=-1,
)
rf.fit(X_train, y_train)
rf_proba = rf.predict_proba(X_test)[:, 1]
rf_auc = roc_auc_score(y_test, rf_proba)
p50_rf = precision_at_k(rf_proba, y_test_r, 50)
p_at_flagged_rf = precision_at_k(rf_proba, y_test_r, n_flagged)

print(f"Baseline flagged {n_flagged} of {len(test_df)} test rows (score > 0)\n")
print(f"{'Method':<20}{'precision@50':>14}{f'precision@{n_flagged}':>16}{'AUC':>8}")
print(f"{'Base rate':<20}{y_test.mean():>14.3f}{'-':>16}{'-':>8}")
print(f"{'Baseline rule':<20}{p50_baseline:>14.3f}{p_at_flagged_baseline:>16.3f}{'-':>8}")
print(f"{'Logistic Regr.':<20}{p50_lr:>14.3f}{p_at_flagged_lr:>16.3f}{lr_auc:>8.3f}")
print(f"{'Random Forest':<20}{p50_rf:>14.3f}{p_at_flagged_rf:>16.3f}{rf_auc:>8.3f}")

Baseline flagged 19 of 4031 test rows (score > 0)

Method                precision@50    precision@19     AUC
Base rate                    0.599               -       -
Baseline rule                0.800           1.000       -
Logistic Regr.               0.840           0.895   0.725
Random Forest                0.840           0.789   0.763


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Random Forest wins on AUC (0.763 vs LR's 0.725) and ties LR on precision@50
(0.84), so it's my pick — with the caveat that at very small k the plain
baseline rule is still the safer, ship-it-today choice.

Top features (RF importances): days_with_impressions (0.14), avg_position
(0.14), impressions_90d (0.13), content_age_days (0.10) — all traffic-history
and age signals. That makes sense for this label: trend_direction is measured
by comparing a page's own recent window to its own earlier window, so pages
with more accumulated history (older, more days with impressions) simply have
more chances to show a measured decline. None of the top features are
"suspiciously perfect" (no single feature above ~14% importance) — no repeat
of the w03 leakage pattern, which is a good sign the honest feature set held.
Notably days_since_last_update (0.033) ranks low, and that turns out to be
exactly where the model's errors cluster (below).

Of the model's top 50 picks, 8 are wrong (predicted high risk, page is
actually stable/growing). All 8 share the same pattern: keyword articles,
recently refreshed (days_since_last_update = 15–20), but with an age/traffic
footprint (content_age_days ~90–139, similar impressions_90d) that matches
declining pages almost exactly. Compare to a correct pick with the same
content_type and similar age (106 days) but days_since_last_update = 106 too
(never refreshed) — same shape everywhere except recency of the last edit.

Concrete example: content_d020d42e7fcc — content_age_days=97,
impressions_90d=2,377, avg_position=2.1, but days_since_last_update=20 (a
recent refresh). The model gave it 0.72 probability of declining, but its
trend_direction is not "down". My read: the model leans hard on age and
traffic footprint (which look identical to a page mid-decline) and
under-weights how recently someone already fixed it — three features that
correlate with "old, established page" are doing most of the work, and the
one feature that would catch "already refreshed" (days_since_last_update)
gets crowded out. A practical fix I'd try next: explicitly downweight or
gate pages with a very recent update, since that's exactly the case a
content editor already knows is handled.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Top 10 Random Forest feature importances:")
print(importances.head(10).to_string())

order = np.argsort(-rf_proba)[:50]
top50 = test_df.iloc[order].copy()
top50["y_true"] = y_test_r.iloc[order].values
top50["rf_proba"] = rf_proba[order]

wrong = top50[top50["y_true"] == 0]
right_stale = top50[(top50["y_true"] == 1)].sort_values("days_since_last_update", ascending=False).head(3)

print(f"\nWrong picks in RF's top 50: {len(wrong)} of 50")
cols = ["content_id", "content_type", "content_age_days", "days_since_last_update",
        "impressions_90d", "avg_position", "rf_proba"]
print("\nWrong (predicted declining, actually stable/growing):")
print(wrong[cols].to_string(index=False))
print("\nFor comparison — correct picks, same content_type/age range, but NOT recently refreshed:")
print(right_stale[cols].to_string(index=False))

Top 10 Random Forest feature importances:
days_with_impressions     0.142080
avg_position              0.139647
impressions_90d           0.133234
content_age_days          0.102162
word_count                0.056973
char_count                0.056221
clicks_90d                0.044172
has_position_data         0.040433
days_since_last_update    0.032861
ctr                       0.031237

Wrong picks in RF's top 50: 8 of 50

Wrong (predicted declining, actually stable/growing):
          content_id    content_type  content_age_days  days_since_last_update  impressions_90d  avg_position  rf_proba
content_0097308f0d12 keyword article               139                      15              204          12.1  0.721682
content_baa7be95655c keyword article                90                      20              114           5.2  0.719030
content_cb30d684acbf keyword article                90                      20              316           6.3  0.717844
content_d020d42e7fcc keyword article

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.